# Topic: SQL Cohort Retention Analysis

## Definition (30-second explanation)
* Cohort retention analysis groups users by the time period they first appeared (like their signup month).
* It then measures what percentage of each cohort returned to be active in subsequent periods (months, weeks, etc.).

## Why Interviewers Ask This
* It is one of the most important metrics for evaluating product health and stickiness over time.
* It tests your ability to structure complex, multi-step SQL queries using CTEs and self-joins.
* It verifies you can translate a business concept into mathematical logic (e.g., calculating period numbers).

## Core Concepts
* **Cohorts:** Users are grouped by a shared starting event, usually their signup month, defined using `DATE_TRUNC()`.
* **Period Number:** The time elapsed (in months or weeks) since the user's signup. Period 0 is the signup month, and retention is always 100% by definition.
* **Cohort Size:** The total count of unique users who joined in that specific cohort.
* **Retention Rate:** The formula is (retained users in period N / cohort size) * 100.

## When to Use
* Any question asking for 'retention rate' or 'what percentage of users came back'.
* Building data for a cohort retention heatmap.
* Analyzing user engagement trends over multiple months or weeks.

## Advantages
* Standardizes user lifecycles, allowing you to fairly compare the behavior of early adopters against newly acquired users.
* Clearly visualizes when the majority of users drop off (e.g., "we lose 50% of users in Month 2").

## Limitations
* Cohort analysis only measures *if* a user came back, not *how much* they engaged or how much revenue they generated.

## Common Comparisons
* **Retained vs. Active:** 'Retained' means they came back in a subsequent period relative to their cohort. 'Active on a specific day' is an instantaneous metric like DAU (Daily Active Users).

## Common Interview Traps
* **Double Counting:** Using `COUNT(*)` instead of `COUNT(DISTINCT user_id)`, which artificially inflates retention if a user logged in twice in one month.
* **Granularity Errors:** Using exact `signup_date` instead of truncating to `cohort_month`, which fragments the cohorts and ruins the analysis.
* **Missing Denominator:** Forgetting to calculate or join back to the original cohort sizes, making it impossible to calculate the final percentage.
* **Off-by-One Errors:** Miscalculating the period. Period 0 should be the signup month, and Period 1 is the following month.

## Python / SQL Syntax (if applicable)
```sql
-- Calculating months elapsed in PostgreSQL
EXTRACT(YEAR FROM age(activity_month, cohort_month)) * 12 
+ EXTRACT(MONTH FROM age(activity_month, cohort_month)) AS period_number
```

## Important Formula (if applicable)
* **Retention Rate** = `(COUNT(DISTINCT retained_user_id) / cohort_size) * 100.0`.

## 45-Second Interview Answer
"To build a cohort retention analysis, I use a three-step CTE approach. First, I assign every user to a cohort, usually by truncating their signup date to the month. Second, I calculate the 'period number' for every activity event by finding the month difference between the activity date and the signup cohort. Finally, I calculate the initial size of each cohort at Period 0, and join that back to the subsequent periods so I can divide the distinct count of returning users by the original cohort size to get the retention percentage."

## Example Questions and Answers:

### Q1. Build a weekly cohort retention table for the first 4 weeks after signup.

**Ideal Interview Answer (MySQL):**
```sql
WITH user_cohorts AS (
    SELECT 
        user_id,
        -- Truncate to the start of the week (Monday)
        DATE_SUB(signup_date, INTERVAL WEEKDAY(signup_date) DAY) AS cohort_week,
        DATE_SUB(activity_date, INTERVAL WEEKDAY(activity_date) DAY) AS activity_week
    FROM user_activity
),
cohort_periods AS (
    SELECT 
        user_id,
        cohort_week,
        -- Calculate weeks elapsed
        TIMESTAMPDIFF(WEEK, cohort_week, activity_week) AS week_number
    FROM user_cohorts
),
cohort_sizes AS (
    SELECT 
        cohort_week, 
        COUNT(DISTINCT user_id) AS total_users
    FROM cohort_periods
    WHERE week_number = 0
    GROUP BY cohort_week
)
SELECT 
    cp.cohort_week,
    cp.week_number,
    cs.total_users AS cohort_size,
    COUNT(DISTINCT cp.user_id) AS retained_users,
    ROUND(100.0 * COUNT(DISTINCT cp.user_id) / cs.total_users, 2) AS retention_pct
FROM cohort_periods cp
JOIN cohort_sizes cs ON cp.cohort_week = cs.cohort_week
WHERE cp.week_number <= 4
GROUP BY cp.cohort_week, cp.week_number, cs.total_users
ORDER BY cp.cohort_week, cp.week_number;
```

**Common Mistakes Candidates Make:**
*   Incorrectly calculating the weekly difference.
*   Forgetting `WHERE week_number <= 4` to limit the output as requested.

**One Likely Interviewer Follow-up:**
"How would this change if `signup_date` and `activity_date` were in two completely different tables?"
*(Answer: The very first CTE would require a `LEFT JOIN` from the `users` table to the `activity` table on `user_id`.)*

### Q2. Find the cohort month with the highest 3-month retention rate.

**Ideal Interview Answer (MySQL):**
```sql
WITH cohorts AS (
    SELECT 
        user_id,
        -- Truncate to the first day of the month
        DATE_FORMAT(signup_date, '%Y-%m-01') AS cohort_month,
        DATE_FORMAT(activity_date, '%Y-%m-01') AS activity_month
    FROM user_activity
),
cohort_periods AS (
    SELECT 
        user_id,
        cohort_month,
        -- Calculate months elapsed
        TIMESTAMPDIFF(MONTH, cohort_month, activity_month) AS period_number
    FROM cohorts
),
cohort_sizes AS (
    SELECT cohort_month, COUNT(DISTINCT user_id) AS cohort_size
    FROM cohort_periods
    WHERE period_number = 0
    GROUP BY cohort_month
)
SELECT 
    cp.cohort_month,
    ROUND(100.0 * COUNT(DISTINCT cp.user_id) / MAX(cs.cohort_size), 2) AS month_3_retention_rate
FROM cohort_periods cp
JOIN cohort_sizes cs ON cp.cohort_month = cs.cohort_month
WHERE cp.period_number = 3
GROUP BY cp.cohort_month
ORDER BY month_3_retention_rate DESC
LIMIT 1;
```

**Common Mistakes Candidates Make:**
*   Filtering for `period_number = 3` *before* calculating the initial cohort sizes. The cohort size must always be calculated based on Period 0.

**One Likely Interviewer Follow-up:**
"Why did you use `MAX(cs.cohort_size)` in your SELECT statement instead of putting it in the `GROUP BY`?"
*(Answer: It's just an alternative syntax. Since cohort size is identical for all rows within a cohort, aggregating it with MAX prevents having to add it to the GROUP BY clause, which some developers find cleaner.)*

### Q3. Compare retention rates for mobile vs desktop users by cohort.

**Ideal Interview Answer (Conceptual SQL Strategy):**
To compare by platform, the `platform` column (mobile vs desktop) must be treated as a secondary grouping dimension throughout the entire query. 
1. In the `cohort_sizes` CTE, group by BOTH `cohort_month` AND `platform`.
2. In the final `SELECT`, `JOIN` on BOTH `cohort_month` AND `platform`. 
3. The final `GROUP BY` must include `cohort_month`, `platform`, and `period_number`.

**Common Mistakes Candidates Make:**
*   Joining only on `cohort_month` in the final step, causing cross-join multiplication between desktop and mobile cohort sizes.

**One Likely Interviewer Follow-up:**
"What if a user signs up on Mobile (Period 0) but retains on Desktop (Period 1)?"
*(Answer: This requires clarifying business logic with the interviewer. Usually, a user is locked into their signup platform's cohort permanently. Therefore, you should derive `platform` only from their signup event, not their subsequent activity events.)*

### Q4. Identify the drop-off point where retention falls below 20% for most cohorts.

**Ideal Interview Answer (Conceptual Strategy):**
Using the final retention table generated in Q1 or Q2, you would wrap it in a final CTE. You can then query this CTE:
```sql
SELECT 
    period_number,
    AVG(retention_pct) as avg_retention_across_cohorts
FROM final_retention_table
GROUP BY period_number
HAVING AVG(retention_pct) < 20.0
ORDER BY period_number ASC
LIMIT 1;
```

**Common Mistakes Candidates Make:**
*   Overcomplicating the SQL by trying to solve it inside the initial retention calculations rather than just wrapping the completed retention table in a CTE and querying the result.

## Practice Questions:

### Q1: 
**Scenario: You are asked to calculate Monthly Cohort Retention. However, instead of one convenient table, the data engineering team has provided you with two strictly normalized tables:**
```sql
-- Table 1: Records when a user was created
CREATE TABLE users (
    user_id INT,
    signup_date DATE
);

-- Table 2: Records every time a user does an action
CREATE TABLE events (
    event_id INT,
    user_id INT,
    event_date DATE
);
```

**Question: If a user signs up in January, but they never log in again to trigger an event, they will exist in the users table but will have zero rows in the events table. How would you structure the very first part of your SQL query (the base CTE) to ensure your Month 0 Cohort Size accurately counts everyone who signed up, even if they never triggered an event? (Please write the SQL for just the first CTE and explain your join logic).**

**Answer (MySQL):**
```sql
WITH user_cohorts AS (
    SELECT 
        u.user_id,
        -- Truncate to Monthly cohort
        DATE_FORMAT(u.signup_date, '%Y-%m-01') AS cohort_month,
        DATE_FORMAT(e.event_date, '%Y-%m-01') AS activity_month
    FROM users u
    -- Critical: LEFT JOIN ensures users with zero events are still counted in Period 0
    LEFT JOIN events e ON u.user_id = e.user_id
)
-- ... proceed with cohort_periods and cohort_sizes CTEs ...
```

**Interview Tips:**
*   **The INNER JOIN Trap:** If you use an `INNER JOIN` (or simply `FROM users, events WHERE...`), users who never logged a secondary event are completely erased from the dataset. Your Period 0 cohort size will be artificially small, meaning your retention percentages will be mathematically impossible/wrong.
*   **Always start from the Source of Truth:** Your `FROM` clause should always be the table that defines the cohort (the `users` table), `LEFT JOIN`ed to the behavior you are measuring.

### Q2: Weekly cohort retention for the first 4 weeks (ClassicModels Schema in MySQL):

```sql
WITH user_activation AS (
    -- Define the cohort by the customer's very first order date
    SELECT customerNumber, DATE(MIN(orderDate)) AS first_order_date
    FROM orders
    GROUP BY customerNumber
),
user_cohorts AS (
    SELECT 
        o.customerNumber,
        -- Truncate first order date to Monday for the cohort week
        DATE_SUB(ua.first_order_date, INTERVAL WEEKDAY(ua.first_order_date) DAY) AS cohort_week,
        -- Truncate subsequent order dates to Monday for the activity week
        DATE_SUB(o.orderDate, INTERVAL WEEKDAY(o.orderDate) DAY) AS activity_week
    FROM user_activation ua
    LEFT JOIN orders o ON ua.customerNumber = o.customerNumber
),
cohort_periods AS (
    SELECT 
        customerNumber,
        cohort_week,
        -- Calculate weeks elapsed between first order and this order
        TIMESTAMPDIFF(WEEK, cohort_week, activity_week) AS week_number
    FROM user_cohorts
),
cohort_sizes AS (
    SELECT 
        cohort_week, 
        COUNT(DISTINCT customerNumber) AS total_users
    FROM cohort_periods
    WHERE week_number = 0
    GROUP BY cohort_week
)
SELECT 
    cp.cohort_week,
    cp.week_number,
    cs.total_users AS cohort_size,
    COUNT(DISTINCT cp.customerNumber) AS retained_users,
    ROUND(100.0 * COUNT(DISTINCT cp.customerNumber) / cs.total_users, 2) AS retention_pct
FROM cohort_periods cp
JOIN cohort_sizes cs ON cp.cohort_week = cs.cohort_week
WHERE cp.week_number <= 4
GROUP BY cp.cohort_week, cp.week_number, cs.total_users
ORDER BY cp.cohort_week, cp.week_number;
```

### Q3: Find the cohort month with the highest 3-month retention rate (ClassicModels  Schema in MySQL):

```sql
WITH user_activation AS (
    SELECT customerNumber, MIN(orderDate) AS first_order_date
    FROM orders
    GROUP BY customerNumber
),
user_cohorts AS (
    SELECT 
        o.customerNumber,
        -- Truncate to the first day of the month
        DATE_FORMAT(ua.first_order_date, '%Y-%m-01') AS cohort_month,
        DATE_FORMAT(o.orderDate, '%Y-%m-01') AS activity_month
    FROM user_activation ua
    LEFT JOIN orders o ON ua.customerNumber = o.customerNumber
),
cohort_periods AS (
    SELECT 
        customerNumber,
        cohort_month,
        -- Calculate months elapsed
        TIMESTAMPDIFF(MONTH, cohort_month, activity_month) AS period_number
    FROM user_cohorts
),
cohort_sizes AS (
    SELECT cohort_month, COUNT(DISTINCT customerNumber) AS cohort_size
    FROM cohort_periods
    WHERE period_number = 0
    GROUP BY cohort_month
)
SELECT 
    cp.cohort_month,
    ROUND(100.0 * COUNT(DISTINCT cp.customerNumber) / MAX(cs.cohort_size), 2) AS month_3_retention_rate
FROM cohort_periods cp
JOIN cohort_sizes cs ON cp.cohort_month = cs.cohort_month
WHERE cp.period_number = 3
GROUP BY cp.cohort_month
ORDER BY month_3_retention_rate DESC
LIMIT 1;
```